In [ ]:
# Required Libraries

import numpy as np
import pandas as pd
import yfinance as yf
from arch import arch_model
from scipy.stats import norm
import nolds
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.gridspec import GridSpec
import warnings
warnings.filterwarnings('ignore')


In [4]:
# Data Intialization

Start = '2010-01-01'
End = pd.Timestamp.today().strftime('%Y-%m-%d')

raw_data = yf.download(
    tickers = ['SPY','^VIX','^VIX3M','^VIX6M'],
    start = Start,
    end = End,
    auto_adjust = True,
    progress = False
)['Close']

raw_data.columns = ['SPY', 'VIX', 'VIX3M', 'VIX6M']
raw_data['SPY_ret'] = np.log(raw_data['SPY']/raw_data['SPY'].shift(1))

df = raw_data.dropna(subset=['SPY','VIX','VIX3M','VIX6M']).copy()


In [5]:
# Variance from VIX 

df['VAR30'] = (df['VIX'] / 100) ** 2
df['VAR93'] = (df['VIX3M'] / 100) ** 2
df['VAR180'] = (df['VIX6M'] / 100) ** 2

df['SLOPE_30_93'] = df['VAR93'] - df['VAR30']
df['SLOPE_93_180'] = df['VAR180'] - df['VAR93']
df['SLOPE_30_180'] = df['VAR180'] - df['VAR30']

# Volatility Regimes
df['REGIME'] = pd.cut(
    df['VIX'],
    bins = [0, 15, 25, np.inf],
    labels = ['Low', 'Medium', 'High']
)


In [10]:
# GARCH
ret = df['SPY_ret'].dropna() * 100

garch = arch_model(
    ret,
    vol = 'GARCH',
    p = 1, q = 1,
    mean = 'Constant',
    dist = 'normal'
)

res = garch.fit(disp = 'off')
print(res.summary())

df.loc[ret.index, 'GARCH_VAR_daily'] = res.conditional_volatility ** 2 / 10000
df['GARCH_VAR_ann'] = df['GARCH_VAR_daily'] * 252

# GARCH Parameters
omega = res.params['omega']
alpha = res.params['alpha[1]']
beta = res.params['beta[1]']
phi = alpha + beta


                     Constant Mean - GARCH Model Results                      
Dep. Variable:                SPY_ret   R-squared:                       0.000
Mean Model:             Constant Mean   Adj. R-squared:                  0.000
Vol Model:                      GARCH   Log-Likelihood:               -5268.44
Distribution:                  Normal   AIC:                           10544.9
Method:            Maximum Likelihood   BIC:                           10570.1
                                        No. Observations:                 4101
Date:                Sun, Apr 26 2026   Df Residuals:                     4100
Time:                        11:10:48   Df Model:                            1
                                Mean Model                                
                 coef    std err          t      P>|t|    95.0% Conf. Int.
--------------------------------------------------------------------------
mu             0.0852  1.176e-02      7.249  4.185e-13 [6.218e-0